# 07 · Gecikme ölçümü (bütün modeller, tek oturum) — eğitim yok, ~10 dk
**Neden:** 06c'de Önerilen-S (h = 64), mimarisi Hibrit ile neredeyse aynı olduğu hâlde CPU'da 58 ms ölçüldü (Hibrit 15 ms).
Ölçümler farklı günlerde/oturumlarda alınmıştı; arka plan yükü ve güç durumu sonucu bozabilir.
Burada bütün modeller aynı oturumda, aynı girdiyle, karışık sırada 3 tur ölçülür. Sonuç `configs\verimlilik_yeniden.json` ve
`excel\07_gecikme.xlsx` dosyalarına yazılır; 06c tekrar çalıştırılınca T13 tablosu yeni değerleri kullanır.
**Ölçümden önce:** diğer programları kapatın, dizüstüyse prize takın, Windows güç modunu "En iyi performans" yapın.

In [ ]:
import sys, os, json, time, warnings, copy
import os; ROOT = os.path.abspath('..')  # repository root (run notebooks from the notebooks/ folder)
SMOKE = False
sys.path.insert(0, ROOT)
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch
from data_utils import *
from models.experiment import *
from models import engine
exp = Experiment(ROOT, smoke=SMOKE)
exp.load_data()
print('Cihaz:', exp.dev, '| W =', exp.cfg['W'], '| CPU iş parçacığı (varsayılan):', torch.get_num_threads())

In [ ]:
# Bütün modeller AYNI oturumda, AYNI girdiyle, karışık sırada 3 tur ölçülür (arka plan yükünün etkisi dağılır).
# Ölçüm: tek pencere (B = 1, W = 256), torch.inference_mode, ısınma 50 + tekrar 300; tur medyanlarının medyanı raporlanır.
# CPU: (i) varsayılan iş parçacığı sayısı, (ii) tek iş parçacığı (gömülü sisteme daha yakın, daha kararlı).
# ⚠️ Ölçüm sırasında başka program (tarayıcı, eğitim, OneDrive senkronu) açık olmasın; dizüstüyse prize takılı olsun.
MODELS = [('final_c', C_NAME), ('ablation_c', 'C: eşit boyut (h=64)'), ('final', 'Hibrit (GRU→Transformer)'),
          ('final', 'LSTM'), ('final', 'GRU'), ('final', 'CNN-LSTM'), ('final', 'Transformer')]
SEED = exp.cfg['FINAL_SEEDS'][0]; ROUNDS = 3; N_WARM, N_REP = 50, 300
NT = torch.get_num_threads()
X = next(iter(exp.datasets(exp.cfg['W'])['test'].values())).X[:exp.cfg['W']].unsqueeze(0)

def lat(m, x, sync):
    ts = []
    with torch.inference_mode():
        for i in range(N_WARM + N_REP):
            if sync: torch.cuda.synchronize()
            t0 = time.perf_counter(); m(x)
            if sync: torch.cuda.synchronize()
            if i >= N_WARM: ts.append((time.perf_counter() - t0) * 1e3)
    return float(np.median(ts)), float(np.percentile(ts, 95))

gpu, cpu = {}, {}
for stage, name in MODELS:
    m, rec = exp.load_model(stage, name, SEED); m.eval()
    gpu[name] = m if exp.dev.type == 'cuda' else None
    cpu[name] = copy.deepcopy(m).cpu().eval()
    print(f'yüklendi: {name:32s} parametre {sum(p.numel() for p in m.parameters()):,}')
rng = np.random.default_rng(0); rows = []
for r in range(ROUNDS):
    order = list(cpu); rng.shuffle(order)
    for name in order:
        row = {'tur': r + 1, 'model': name}
        if gpu[name] is not None:
            row['GPU_ms_medyan'], row['GPU_ms_p95'] = lat(gpu[name], X, True)
        torch.set_num_threads(NT); row['CPU_ms_medyan'], row['CPU_ms_p95'] = lat(cpu[name], X.cpu(), False)
        torch.set_num_threads(1);  row['CPU1_ms_medyan'], row['CPU1_ms_p95'] = lat(cpu[name], X.cpu(), False)
        torch.set_num_threads(NT)
        rows.append(row); print(f"tur {r + 1} · {name:32s} " + ' · '.join(f'{k} {v:.2f}' for k, v in row.items() if k.endswith('medyan')))
raw = pd.DataFrame(rows)

In [ ]:
agg = raw.groupby('model').median(numeric_only=True).drop(columns='tur')
agg['tur arası yayılım CPU (%)'] = (raw.groupby('model').CPU_ms_medyan.agg(lambda s: (s.max() - s.min()) / s.median() * 100))
agg['parametre'] = [sum(p.numel() for p in cpu[n].parameters()) for n in agg.index]
agg['MAC_pencere'] = [engine.count_macs(cpu[n], X.cpu()) for n in agg.index]
agg['CPU_iş_parçacığı'] = NT
agg = agg.reindex([n for _, n in MODELS]).reset_index()
print(agg.round(3).to_string(index=False))
agg.to_json(exp.p['configs'] / 'verimlilik_yeniden.json', orient='records', force_ascii=False, indent=1)
with pd.ExcelWriter(exp.p['excel'] / '07_gecikme.xlsx') as xw:
    agg.to_excel(xw, sheet_name='Özet', index=False); raw.to_excel(xw, sheet_name='Turlar', index=False)
print('✅ configs/verimlilik_yeniden.json — 06c yeniden çalıştırılınca T13 bu değerleri kullanır')

### Nasıl okunur
- **Tur arası yayılım** < %10 ise ölçüm kararlıdır. Daha büyükse arka plan yükü var; ölçümü tekrarlayın.
- Makalede önerilen: **parametre + MAC** (cihazdan bağımsız) ve **tek iş parçacıklı CPU gecikmesi** (en kararlı) + donanım bilgisi (`configs\ortam.json`).
- Beklenti: Önerilen-S ≈ Hibrit (aynı mimari); Önerilen (h = 128) ikisinden yavaş.